# Enterprise ML Pipelines: SAS to Databricks Migration

This advanced notebook keeps the code readable while introducing production ideas: distributed data generation, cross-validation, feature expansion, and model metadata.


## 1. Import Libraries

The imports are explicit so beginners can see where each class comes from.


In [ ]:
import json
import warnings
from datetime import datetime

from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import PolynomialExpansion, StandardScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Enterprise regression notebook")
print(f"Spark version: {spark.version}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 2. Create Distributed Training Data

This uses Spark columns instead of local Python loops. That is the main mindset shift from SAS to Databricks: let Spark create and transform data across the cluster.


In [ ]:
row_count = 100_000

houses = (
    spark.range(row_count)
    .withColumn("square_footage", (F.rand(seed=42) * 5000 + 1000).cast("double"))
    .withColumn("bedrooms", (F.rand(seed=43) * 5 + 1).cast("int"))
    .withColumn("bathrooms", (F.rand(seed=44) * 3 + 1).cast("double"))
    .withColumn("age", (F.rand(seed=45) * 100).cast("int"))
    .withColumn("garage_spaces", (F.rand(seed=46) * 4).cast("int"))
    .withColumn("has_pool", (F.rand(seed=47) * 2).cast("int"))
    .withColumn("lot_size_sqft", (F.rand(seed=48) * 45000 + 5000).cast("double"))
)

houses = houses.withColumn(
    "price",
    120 * F.col("square_footage")
    + 45000 * F.col("bedrooms")
    + 30000 * F.col("bathrooms")
    - 1500 * F.col("age")
    + 15000 * F.col("garage_spaces")
    + 75000 * F.col("has_pool")
    + 2 * F.col("lot_size_sqft")
    + F.rand(seed=49) * 100000,
).drop("id")

houses = houses.cache()

print(f"Rows: {houses.count():,}")
print(f"Partitions: {houses.rdd.getNumPartitions()}")
houses.show(5)


## 3. SAS Reference: Large Data Generation

SAS can generate rows with a `DATA` step, but it usually runs on one server. Spark can distribute row creation and transformations across a cluster.

```sas
DATA large_house_data;
  DO row_id = 1 TO 100000;
    square_footage = 1000 + RAND('UNIFORM') * 5000;
    bedrooms = CEIL(RAND('UNIFORM') * 5);
    bathrooms = 1 + RAND('UNIFORM') * 3;
    age = FLOOR(RAND('UNIFORM') * 100);
    price = 120*square_footage + 45000*bedrooms + 30000*bathrooms - 1500*age;
    OUTPUT;
  END;
RUN;
```


## 4. Visualize a Sample from Distributed Data

Large Spark DataFrames should not be fully converted to pandas. For plotting, take a small sample first, then use Matplotlib on that sample.


In [ ]:
sample_data = houses.select("square_footage", "price").sample(fraction=0.01, seed=42).limit(1000).toPandas()

plt.figure(figsize=(8, 5))
plt.scatter(sample_data["square_footage"], sample_data["price"], alpha=0.4)
plt.title("Sample of Distributed House Data")
plt.xlabel("Square Footage")
plt.ylabel("Price")
plt.grid(True)
plt.show()


## 5. Build a Cross-Validated Pipeline

Cross-validation trains several versions of the model and chooses the best settings using a metric. Here we optimize for R2.


In [ ]:
train_data, test_data = houses.randomSplit([0.7, 0.3], seed=42)

feature_columns = [
    "square_footage",
    "bedrooms",
    "bathrooms",
    "age",
    "garage_spaces",
    "has_pool",
    "lot_size_sqft",
]
target_column = "price"

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
regression = LinearRegression(featuresCol="scaled_features", labelCol=target_column)

pipeline = Pipeline(stages=[assembler, scaler, regression])

parameter_grid = (
    ParamGridBuilder()
    .addGrid(regression.regParam, [0.0, 0.01, 0.1])
    .addGrid(regression.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

r2_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="r2")

cross_validator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=parameter_grid,
    evaluator=r2_evaluator,
    numFolds=3,
    parallelism=2,
)

print(f"Parameter combinations: {len(parameter_grid)}")
print("Training cross-validated model...")

started_at = datetime.now()
cross_validated_model = cross_validator.fit(train_data)
elapsed_seconds = (datetime.now() - started_at).total_seconds()

best_regression = cross_validated_model.bestModel.stages[-1]

print(f"Training finished in {elapsed_seconds:.1f} seconds")
print(f"Best regParam: {best_regression.getRegParam()}")
print(f"Best elasticNetParam: {best_regression.getElasticNetParam()}")


## 6. SAS Reference: Manual Parameter Search

SAS users often compare parameter choices manually across multiple procedure runs. PySpark `CrossValidator` automates this pattern.

```sas
PROC REG DATA=train_data;
  MODEL price = square_footage bedrooms bathrooms age / SELECTION=NONE;
RUN;
QUIT;

PROC GLMSELECT DATA=train_data;
  MODEL price = square_footage bedrooms bathrooms age garage_spaces has_pool lot_size_sqft
    / SELECTION=LASSO(CHOOSE=CV);
RUN;
```


## 7. Evaluate the Best Model

Keep evaluation code short and repeatable. These values can later be logged to MLflow or another model registry.


In [ ]:
test_predictions = cross_validated_model.transform(test_data)

rmse_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="mae")

best_test_rmse = rmse_evaluator.evaluate(test_predictions)
best_test_mae = mae_evaluator.evaluate(test_predictions)
best_test_r2 = r2_evaluator.evaluate(test_predictions)

print("Best model test metrics")
print(f"RMSE: {best_test_rmse:,.0f}")
print(f"MAE:  {best_test_mae:,.0f}")
print(f"R2:   {best_test_r2:.4f}")


## 8. Visualize Best Model Predictions

This plot checks whether the cross-validated model predicts close to the real test values. It uses a small sample so the chart stays readable.


In [ ]:
actual_vs_predicted = test_predictions.select(target_column, "prediction").limit(1000).toPandas()

min_price = min(actual_vs_predicted[target_column].min(), actual_vs_predicted["prediction"].min())
max_price = max(actual_vs_predicted[target_column].max(), actual_vs_predicted["prediction"].max())

plt.figure(figsize=(6, 6))
plt.scatter(actual_vs_predicted[target_column], actual_vs_predicted["prediction"], alpha=0.35)
plt.plot([min_price, max_price], [min_price, max_price], color="red", label="Perfect prediction")
plt.title("Cross-Validated Model: Actual vs Predicted")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.legend()
plt.grid(True)
plt.show()


## 9. Add Polynomial Features

Polynomial features are useful when the business relationship is curved or when feature interactions matter. They are more expensive, so compare them with the simpler model.


In [ ]:
polynomial_assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
polynomial_expansion = PolynomialExpansion(degree=2, inputCol="features", outputCol="polynomial_features")
polynomial_scaler = StandardScaler(inputCol="polynomial_features", outputCol="scaled_polynomial_features", withMean=True, withStd=True)
polynomial_regression = LinearRegression(
    featuresCol="scaled_polynomial_features",
    labelCol=target_column,
    regParam=0.01,
)

polynomial_pipeline = Pipeline(stages=[
    polynomial_assembler,
    polynomial_expansion,
    polynomial_scaler,
    polynomial_regression,
])

print("Training polynomial model...")
polynomial_model = polynomial_pipeline.fit(train_data)

polynomial_predictions = polynomial_model.transform(test_data)
polynomial_rmse = rmse_evaluator.evaluate(polynomial_predictions)
polynomial_r2 = r2_evaluator.evaluate(polynomial_predictions)

original_feature_count = len(feature_columns)
polynomial_feature_count = int((original_feature_count + 1) * (original_feature_count + 2) / 2 - 1)

print(f"Original feature count: {original_feature_count}")
print(f"Polynomial feature count: {polynomial_feature_count}")
print(f"Polynomial RMSE: {polynomial_rmse:,.0f}")
print(f"Polynomial R2: {polynomial_r2:.4f}")


## 10. Visualize Production Model Metrics

These charts compare the simpler cross-validated model with the polynomial model. A production team can use this to decide whether extra complexity is worth it.


In [ ]:
model_names = ["Cross-Validated Linear", "Polynomial"]
rmse_scores = [best_test_rmse, polynomial_rmse]
r2_scores = [best_test_r2, polynomial_r2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(model_names, rmse_scores)
axes[0].set_title("RMSE Comparison")
axes[0].set_ylabel("RMSE")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y")

axes[1].bar(model_names, r2_scores)
axes[1].set_title("R2 Comparison")
axes[1].set_ylabel("R2 Score")
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y")

plt.tight_layout()
plt.show()


## 11. Capture Model Metadata

Production teams need enough metadata to explain what was trained, when it was trained, and how it performed.


In [ ]:
model_metadata = {
    "model_name": "sas_to_databricks_regression",
    "created_at": datetime.now().isoformat(),
    "training_rows": train_data.count(),
    "test_rows": test_data.count(),
    "features": feature_columns,
    "label": target_column,
    "best_model": {
        "rmse": float(best_test_rmse),
        "mae": float(best_test_mae),
        "r2": float(best_test_r2),
        "regParam": best_regression.getRegParam(),
        "elasticNetParam": best_regression.getElasticNetParam(),
    },
    "polynomial_model": {
        "degree": 2,
        "feature_count": polynomial_feature_count,
        "rmse": float(polynomial_rmse),
        "r2": float(polynomial_r2),
    },
}

print(json.dumps(model_metadata, indent=2))


## 12. SAS Reference: Saving Model Outputs

SAS workflows often save scored rows and model reports as datasets or ODS output. The Databricks version stores metadata as a Python dictionary that can later be logged to MLflow or saved as JSON.

```sas
PROC REG DATA=train_data OUTEST=model_parameters;
  MODEL price = square_footage bedrooms bathrooms age garage_spaces has_pool lot_size_sqft;
RUN;
QUIT;

PROC SCORE DATA=test_data SCORE=model_parameters OUT=scored_test TYPE=PARMS;
  VAR square_footage bedrooms bathrooms age garage_spaces has_pool lot_size_sqft;
RUN;
```


## 13. Enterprise Migration Notes

| Need | SAS Pattern | Databricks Pattern |
| --- | --- | --- |
| Scale beyond one machine | Large SAS server | Spark cluster |
| Repeatable training | Stored programs/macros | ML pipelines |
| Parameter search | Manual PROC runs | `CrossValidator` |
| Model scoring | `PROC SCORE` | `model.transform()` |
| Model tracking | External documentation | MLflow or registry metadata |

Recommended migration path:
1. Start with a simple baseline model.
2. Match SAS output on a small sample.
3. Move feature engineering into a PySpark pipeline.
4. Add cross-validation after the baseline is trusted.
5. Save model metrics and metadata with every training run.
